In [0]:
dbutils.widgets.text("catalog", "dbr_dev_ua5816bd")
dbutils.widgets.text("schema_prefix", "elina_sharabura")

CATALOG = dbutils.widgets.get("catalog")
SCHEMA_PREFIX = dbutils.widgets.get("schema_prefix")

BRONZE_SCHEMA = f"{CATALOG}.{SCHEMA_PREFIX}_bronze"
SILVER_SCHEMA = f"{CATALOG}.{SCHEMA_PREFIX}_silver"
GOLD_SCHEMA = f"{CATALOG}.{SCHEMA_PREFIX}_gold"

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {BRONZE_SCHEMA}.order_items (
    order_id            INT,
    product_id          INT,
    unit_price_usd      DOUBLE,
    quantity            INT,
    line_total_usd      DOUBLE,
    _rescued_data       STRING,
    source_filename     STRING,
    ingestion_timestamp TIMESTAMP,
    load_date           DATE
)
COMMENT 'Raw order line items ingested from raw_data/order_items via Auto Loader'
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {SILVER_SCHEMA}.orders (
    order_id            INT       NOT NULL,
    customer_id         INT       NOT NULL,
    order_time          TIMESTAMP NOT NULL,
    payment_method      STRING,
    discount_pct        INT,
    subtotal_usd        DOUBLE,
    total_usd           DOUBLE,
    country             STRING,
    device              STRING,
    source              STRING,
    source_filename     STRING,
    ingestion_timestamp TIMESTAMP,
    processed_at        TIMESTAMP
)
CLUSTER BY (order_id)
COMMENT 'Validated order headers, one row per order'
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {SILVER_SCHEMA}.order_items (
    order_id            INT    NOT NULL,
    line_number         INT    NOT NULL COMMENT 'Sequential line position within an order; the source has no line id and the same product can appear twice in one order',
    product_id          INT    NOT NULL,
    unit_price_usd      DOUBLE,
    quantity            INT,
    line_total_usd      DOUBLE,
    source_filename     STRING,
    ingestion_timestamp TIMESTAMP,
    processed_at        TIMESTAMP
)
CLUSTER BY (order_id)
COMMENT 'Validated order line items, one row per line within an order'
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {GOLD_SCHEMA}.dim_date (
    date_key     INT     NOT NULL COMMENT 'Date in yyyyMMdd format, e.g. 20250314',
    full_date    DATE    NOT NULL,
    year         INT,
    quarter      INT,
    month        INT,
    month_name   STRING,
    day_of_month INT,
    day_name     STRING,
    is_weekend   BOOLEAN,
    CONSTRAINT pk_dim_date PRIMARY KEY (date_key)
)
COMMENT 'Calendar dimension, one row per day across the order date range'
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {GOLD_SCHEMA}.dim_customer (
    customer_sk      BIGINT    NOT NULL COMMENT 'Surrogate key of a customer version, reused from silver',
    customer_id      INT       NOT NULL COMMENT 'Business key; one customer can have several versions',
    customer_name    STRING,
    email            STRING,
    country          STRING    COMMENT 'Customer home country (may differ from the order country)',
    age              INT,
    loyalty_tier     STRING,
    signup_date      DATE,
    marketing_opt_in BOOLEAN,
    effective_from   TIMESTAMP,
    effective_to     TIMESTAMP,
    is_current       BOOLEAN,
    CONSTRAINT pk_dim_customer PRIMARY KEY (customer_sk)
)
COMMENT 'Customer dimension with full SCD2 history, one row per customer version'
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {GOLD_SCHEMA}.dim_product (
    product_id   INT           NOT NULL,
    product_name STRING,
    category     STRING,
    price_usd    DECIMAL(12,2) COMMENT 'Current list price; the price actually paid is in fact_order_items',
    CONSTRAINT pk_dim_product PRIMARY KEY (product_id)
)
COMMENT 'Product dimension (SCD1, current state only)'
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {GOLD_SCHEMA}.fact_order_items (
    order_id            INT           NOT NULL,
    line_number         INT           NOT NULL,
    date_key            INT           NOT NULL,
    customer_sk         BIGINT        NOT NULL COMMENT 'Customer version valid at order time',
    product_id          INT           NOT NULL,
    country             STRING        COMMENT 'Order country; used for row-level security',
    payment_method      STRING,
    device              STRING,
    source              STRING,
    quantity            INT,
    unit_price_usd      DECIMAL(12,2) COMMENT 'Price actually paid at order time',
    discount_pct        INT           COMMENT 'Order-level discount applied to every line',
    gross_amount_usd    DECIMAL(12,2) COMMENT 'unit_price_usd * quantity, before discount',
    discount_amount_usd DECIMAL(12,2),
    net_amount_usd      DECIMAL(12,2) COMMENT 'Revenue after discount; sums to the order total',
    processed_at        TIMESTAMP,
    CONSTRAINT pk_fact_order_items PRIMARY KEY (order_id, line_number),
    CONSTRAINT fk_fact_date     FOREIGN KEY (date_key)    REFERENCES {GOLD_SCHEMA}.dim_date (date_key),
    CONSTRAINT fk_fact_customer FOREIGN KEY (customer_sk) REFERENCES {GOLD_SCHEMA}.dim_customer (customer_sk),
    CONSTRAINT fk_fact_product  FOREIGN KEY (product_id)  REFERENCES {GOLD_SCHEMA}.dim_product (product_id)
)
CLUSTER BY (date_key)
COMMENT 'Sales fact at order line grain: one row per product line within an order'
""")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {GOLD_SCHEMA}.agg_daily_sales (
    date_key            INT           NOT NULL,
    country             STRING        NOT NULL COMMENT 'Order country; used for row-level security',
    category            STRING        NOT NULL,
    orders_count        BIGINT,
    items_count         BIGINT,
    quantity            BIGINT,
    gross_amount_usd    DECIMAL(14,2),
    discount_amount_usd DECIMAL(14,2),
    net_amount_usd      DECIMAL(14,2),
    processed_at        TIMESTAMP,
    CONSTRAINT pk_agg_daily_sales PRIMARY KEY (date_key, country, category),
    CONSTRAINT fk_agg_date FOREIGN KEY (date_key) REFERENCES {GOLD_SCHEMA}.dim_date (date_key)
)
COMMENT 'Daily sales by order country and product category, pre-aggregated for dashboards'
""")